In [1]:
import os
import time
import math
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm.autonotebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# 深度学习库
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 机器学习库
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb

/tmp/ipykernel_3678947/3916166659.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
2025-12-04 18:10:23.423919: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-04 18:10:23.492864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 18:10:25.342483: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due t

In [2]:
X_3d = np.load('./train_data/train_data_X_3d.npy')
y = np.load('./train_data/train_data_y.npy')

In [3]:
X_3d.shape

(78672, 7, 15)

In [4]:
y.shape

(78672,)

In [5]:
# 划分数据集
print("划分数据集...")
test_size = 24 * 365 * 2  # 最后2年作为测试集

X_train, X_test = X_3d[:-test_size], X_3d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

# val_size = int(len(X_train) * 0.25)
# X_train, X_val = X_train[:-val_size], X_train[-val_size:]
# y_train_split, y_val = y_train[:-val_size], y_train[-val_size:]

# print(f"训练集: {len(y_train_split)} 样本")
# print(f"验证集: {len(y_val)} 样本")
# print(f"测试集: {len(y_test)} 样本\n")

# results = {}

划分数据集...


In [6]:
# ==================== PyTorch版本 ====================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

class TransformerTrainerPyTorch:
    """
    Transformer模型训练类 - PyTorch版本
    
    参数:
        seq_len: 时间序列长度 (默认7)
        n_features: 特征数量 (默认15)
        d_model: 模型维度 (默认128)
        nhead: 注意力头数量 (默认4)
        num_layers: Transformer层数 (默认2)
        dim_feedforward: 前馈网络维度 (默认256)
        dropout: Dropout比例 (默认0.1)
        learning_rate: 学习率 (默认0.001)
    """
    
    def __init__(self, seq_len=7, n_features=15, d_model=128, nhead=4, 
                 num_layers=2, dim_feedforward=256, dropout=0.1, learning_rate=0.01):
        self.seq_len = seq_len
        self.n_features = n_features
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.learning_rate = learning_rate
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        self.train_losses = []
        self.val_losses = []
        self.train_maes = []
        self.val_maes = []
        
        print(f"初始化PyTorch Transformer模型:")
        print(f"  设备: {self.device}")
        print(f"  输入形状: ({seq_len}, {n_features})")
        print(f"  模型维度: {d_model}, 注意力头: {nhead}")
        print(f"  前馈维度: {dim_feedforward}, Transformer层数: {num_layers}")
        print(f"  学习率: {learning_rate}\n")
    
    def _build_model(self):
        """构建Transformer模型"""
        
        class TransformerModel(nn.Module):
            def __init__(self, n_features, d_model, nhead, num_layers, 
                         dim_feedforward, dropout, seq_len):
                super(TransformerModel, self).__init__()
                
                # 输入投影层
                self.input_projection = nn.Linear(n_features, d_model)
                
                # 位置编码
                self.pos_encoder = self._generate_positional_encoding(seq_len, d_model)
                
                # Transformer编码器
                encoder_layer = nn.TransformerEncoderLayer(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                    batch_first=True
                )
                
                self.transformer_encoder = nn.TransformerEncoder(
                    encoder_layer,
                    num_layers=num_layers
                )
                
                # 输出层
                self.fc = nn.Sequential(
                    nn.Linear(d_model, 128),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.Linear(128, 64),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.Linear(64, 1)
                )
            
            def _generate_positional_encoding(self, seq_len, d_model):
                """生成位置编码"""
                position = torch.arange(seq_len).unsqueeze(1)
                div_term = torch.exp(torch.arange(0, d_model, 2) * 
                                     (-np.log(10000.0) / d_model))
                
                pe = torch.zeros(seq_len, d_model)
                pe[:, 0::2] = torch.sin(position * div_term)
                pe[:, 1::2] = torch.cos(position * div_term)
                
                return pe.unsqueeze(0)  # (1, seq_len, d_model)
            
            def forward(self, x):
                # x: (batch, seq_len, n_features)
                x = self.input_projection(x)  # (batch, seq_len, d_model)
                
                # 添加位置编码
                x = x + self.pos_encoder.to(x.device)
                
                # Transformer编码
                x = self.transformer_encoder(x)  # (batch, seq_len, d_model)
                
                # 全局平均池化
                x = x.mean(dim=1)  # (batch, d_model)
                
                # 输出
                x = self.fc(x)  # (batch, 1)
                
                return x.squeeze()
        
        model = TransformerModel(
            self.n_features, self.d_model, self.nhead, self.num_layers,
            self.dim_feedforward, self.dropout, self.seq_len
        )
        
        return model.to(self.device)
    
    def fit(self, X_train, y_train, validation_split=0.1, epochs=50, batch_size=64, verbose=1):
        """
        训练模型
        
        参数:
            X_3d: 输入数据, shape=(samples, seq_len, n_features)
            y: 目标变量, shape=(samples,)
            validation_split: 验证集比例
            epochs: 训练轮数
            batch_size: 批次大小
            verbose: 显示训练过程 (0=静默, 1=详细)
        """
        print("="*70)
        print("开始训练 PyTorch Transformer 模型")
        print("="*70)
        
        # 数据标准化
        print("标准化数据...")
        X_scaled = self.scaler_X.fit_transform(
            X_train.reshape(-1, X_train.shape[-1])
        ).reshape(X_train.shape)
        
        y_scaled = self.scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        
        # 划分训练集和验证集
        X_train, X_val, y_train, y_val = train_test_split(
            X_scaled, y_scaled, test_size=validation_split, random_state=42
        )
        # val_size = int(len(X_train) * 0.25)
        # X_train, X_val = X_scaled[:-val_size], X_scaled[-val_size:]
        # y_train, y_val = y_scaled[:-val_size], y_scaled[-val_size:]
        
        # 转换为PyTorch张量
        X_train_tensor = torch.FloatTensor(X_train)
        y_train_tensor = torch.FloatTensor(y_train)
        X_val_tensor = torch.FloatTensor(X_val)
        y_val_tensor = torch.FloatTensor(y_val)
        
        # 创建数据加载器
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        
        # 构建模型
        print("构建模型...")
        self.model = self._build_model()
        
        # 显示模型结构
        if verbose > 0:
            print("\n模型结构:")
            print(self.model)
            total_params = sum(p.numel() for p in self.model.parameters())
            trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
            print(f"\n总参数量: {total_params:,}")
            print(f"可训练参数量: {trainable_params:,}\n")
        
        # 损失函数和优化器
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, verbose=verbose > 0
        )
        
        # 训练循环
        print(f"开始训练 (epochs={epochs}, batch_size={batch_size})...")
        best_val_loss = float('inf')
        patience_counter = 0
        patience = 15
        
        for epoch in range(epochs):
            # 训练阶段
            self.model.train()
            train_loss = 0
            train_mae = 0
            
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                train_mae += torch.abs(outputs - y_batch).mean().item()
            
            train_loss /= len(train_loader)
            train_mae /= len(train_loader)
            
            # 验证阶段
            self.model.eval()
            val_loss = 0
            val_mae = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(self.device)
                    y_batch = y_batch.to(self.device)
                    
                    outputs = self.model(X_batch)
                    loss = criterion(outputs, y_batch)
                    
                    val_loss += loss.item()
                    val_mae += torch.abs(outputs - y_batch).mean().item()
            
            val_loss /= len(val_loader)
            val_mae /= len(val_loader)
            
            # 记录历史
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_maes.append(train_mae)
            self.val_maes.append(val_mae)
            
            # 学习率调度
            scheduler.step(val_loss)
            
            # 早停检查
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            # 打印进度
            if verbose > 0 and (epoch + 1) % 5 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] - "
                      f"Train Loss: {train_loss:.6f}, Train MAE: {train_mae:.6f} | "
                      f"Val Loss: {val_loss:.6f}, Val MAE: {val_mae:.6f}")
            
            # 早停
            if patience_counter >= patience:
                print(f"\n早停触发，在第 {epoch+1} 轮停止训练")
                break
        
        print("\n训练完成!")
        self._print_training_summary()
        
        return self
    
    def _print_training_summary(self):
        """打印训练摘要"""
        print("\n" + "="*70)
        print("训练摘要:")
        print("="*70)
        print(f"最终训练损失 (MSE): {self.train_losses[-1]:.6f}")
        print(f"最终验证损失 (MSE): {self.val_losses[-1]:.6f}")
        print(f"最终训练 MAE: {self.train_maes[-1]:.6f}")
        print(f"最终验证 MAE: {self.val_maes[-1]:.6f}")
        print(f"训练轮数: {len(self.train_losses)}")
        print("="*70 + "\n")
    
    def predict(self, X_3d):
        """预测"""
        self.model.eval()
        
        X_scaled = self.scaler_X.transform(
            X_3d.reshape(-1, X_3d.shape[-1])
        ).reshape(X_3d.shape)
        
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        
        with torch.no_grad():
            predictions_scaled = self.model(X_tensor).cpu().numpy()
        
        predictions = self.scaler_y.inverse_transform(
            predictions_scaled.reshape(-1, 1)
        ).flatten()
        
        return predictions
    
    def evaluate(self, X_3d, y):
        """评估模型"""
        predictions = self.predict(X_3d)
        
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        mape = np.mean(np.abs((y - predictions) / y)) * 100
        
        print("="*70)
        print("模型评估结果 (PyTorch版本):")
        print("="*70)
        print(f"MAE:  {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAPE: {mape:.2f}%")
        print("="*70 + "\n")
        
        return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
    
    def model_save(self):
        torch.save(self.model.state_dict(), './models/model_weights.pth')

In [9]:
# 加载数据
X_3d = np.load(f'./train_data/train_data_X_3d.npy')
y = np.load(f'./train_data/train_data_y.npy')

# 划分数据集
print("划分数据集...")
test_size = 24 * 365 * 1  # 最后2年作为测试集

X_train, X_test = X_3d[:-test_size], X_3d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

划分数据集...


In [10]:
# PyTorch版本
pytorch_model = TransformerTrainerPyTorch(
    seq_len=7,
    n_features=15,
    d_model=128,
    nhead=4,
    num_layers=2
)
pytorch_model.fit(X_train, y_train, epochs=100, batch_size=64)
predictions = pytorch_model.predict(X_test)
results = pytorch_model.evaluate(X_test, y_test)

初始化PyTorch Transformer模型:
  设备: cuda
  输入形状: (7, 15)
  模型维度: 128, 注意力头: 4
  前馈维度: 256, Transformer层数: 2
  学习率: 0.01

开始训练 PyTorch Transformer 模型
标准化数据...
构建模型...

模型结构:
TransformerModel(
  (input_projection): Linear(in_features=15, out_features=128, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Sequential(
  

In [11]:
test_size = 24 * 365 * 1   # 最后2年作为测试集

X_train, X_test = X_3d[:-test_size], X_3d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]
predictions = pytorch_model.predict(X_test)
results = pytorch_model.evaluate(X_test, y_test)

模型评估结果 (PyTorch版本):
MAE:  2487.8010
RMSE: 3608.0717
MAPE: 4.64%

